# 2.1.5 把年龄区间编码为类别

`Your age` 的原文是 `15 to 18`、`19 to 25` 这样的区间，不是精确岁数。

不要使用 `to_numeric`，不要取区间端点或中点当成年龄。健身水平 `Unfit` 到 `Perfect` 也是类别，不要归一化成连续测量。划分时按年龄区间分层。

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

df = pd.read_csv("健康咨询客户数据集.csv")
df.columns = df.columns.str.strip()
df = df.drop(columns=[c for c in ["Timestamp", "Your name"] if c in df.columns])
print(df["Your age"].value_counts(dropna=False))
df = df.dropna().drop_duplicates().copy()

age_order = ["15 to 18", "19 to 25", "26 to 30", "30 to 40", "40 and above"]
# 保持区间原文，并按上面的顺序记为有序类别。空白处不要写成 pd.to_numeric。
df["Your age"] = pd.Categorical(df["Your age"].astype(str), categories=__________, ordered=True)
assert not pd.api.types.is_numeric_dtype(df["Your age"])

fitness_col = "How do you describe your current level of fitness ?"
fitness_order = ["Unfit", "Average", "Good", "Very good", "Perfect"]
df[fitness_col] = pd.Categorical(df[fitness_col].astype(str), categories=fitness_order, ordered=True)

frequency = df["How often do you exercise?"].astype(str).value_counts()
plt.figure(figsize=(8, 6))
plt.pie(frequency.values, labels=frequency.index, autopct="%1.1f%%", startangle=90)
plt.title("Exercise frequency")
plt.savefig("2.1.5_exercise_frequency.png", dpi=100, bbox_inches="tight")
plt.close()

y = df["Your age"].astype(str)
X = df.drop(columns=["Your age"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=__________)
output = X.copy()
output["target"] = y
output["age_interval"] = y
output["split"] = np.where(output.index.isin(X_train.index), "train", "test")
assert not output["target"].str.fullmatch(r"\d+").any()
output.to_csv("2.1.5_cleaned_data.csv", index=False, encoding="utf-8-sig")
print("已保存年龄区间，没有转换成精确年龄。")
print(output["age_interval"].value_counts())
